# Market-Neutral Momentum Strategy
A compact, reproducible quantitative research notebook using liquid US sector ETFs.

The implementation uses strict temporal ordering, monthly rebalancing, inverse-volatility weighting, volatility targeting and turnover-dependent transaction costs.

In [ ]:
!pip -q install yfinance


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf


## Strategy implementation

In [ ]:
import numpy as np
import pandas as pd

TRADING_DAYS = 252


def momentum_signal(prices: pd.DataFrame, lookback: int = 126, skip: int = 21) -> pd.DataFrame:
    """Medium-term momentum: return from t-lookback to t-skip."""
    if lookback <= skip:
        raise ValueError("lookback must be greater than skip")
    return prices.shift(skip) / prices.shift(lookback) - 1.0


def rolling_vol(returns: pd.DataFrame, window: int = 63) -> pd.DataFrame:
    """Annualized trailing volatility for each asset."""
    return returns.rolling(window).std() * np.sqrt(TRADING_DAYS)


def _rebalance_dates(index: pd.DatetimeIndex, frequency: str = "monthly") -> pd.DatetimeIndex:
    if frequency == "daily":
        return index
    if frequency != "monthly":
        raise ValueError("frequency must be 'daily' or 'monthly'")
    s = pd.Series(index=index, data=np.arange(len(index)))
    return s.groupby(index.to_period("M")).tail(1).index


def cross_sectional_weights(
    prices: pd.DataFrame,
    lookback: int = 126,
    skip: int = 21,
    vol_window: int = 63,
    long_frac: float = 0.25,
    short_frac: float = 0.25,
    rebalance_frequency: str = "monthly",
) -> pd.DataFrame:
    """
    Build dollar-neutral long/short weights.

    At each rebalance date, rank assets by medium-term momentum, go long the
    strongest names and short the weakest names, and inverse-volatility weight
    positions within each side. Between rebalances, weights are held constant.
    """
    if prices.shape[1] < 4:
        raise ValueError("At least four assets are required")

    returns = prices.pct_change()
    signal = momentum_signal(prices, lookback, skip)
    vol = rolling_vol(returns, vol_window).replace(0, np.nan)

    weights = pd.DataFrame(np.nan, index=prices.index, columns=prices.columns, dtype=float)
    rebal_dates = set(_rebalance_dates(prices.index, rebalance_frequency))

    for dt in prices.index:
        if dt not in rebal_dates:
            continue

        s = signal.loc[dt].dropna()
        v = vol.loc[dt].reindex(s.index).dropna()
        s = s.reindex(v.index)
        n = len(s)
        if n < 4:
            continue

        k_long = max(1, int(np.ceil(n * long_frac)))
        k_short = max(1, int(np.ceil(n * short_frac)))
        longs = s.nlargest(k_long).index
        shorts = s.nsmallest(k_short).index

        row = pd.Series(0.0, index=prices.columns)
        invv_l = (1.0 / v.loc[longs]).replace([np.inf, -np.inf], np.nan).dropna()
        invv_s = (1.0 / v.loc[shorts]).replace([np.inf, -np.inf], np.nan).dropna()

        if len(invv_l):
            row.loc[invv_l.index] = 0.5 * invv_l / invv_l.sum()
        if len(invv_s):
            row.loc[invv_s.index] = -0.5 * invv_s / invv_s.sum()
        weights.loc[dt] = row

    return weights.ffill().fillna(0.0)


def apply_vol_target(
    raw_strategy_returns: pd.Series,
    base_weights: pd.DataFrame,
    target_vol: float = 0.10,
    vol_window: int = 63,
    max_leverage: float = 2.0,
):
    """Scale portfolio exposure using only trailing realized strategy volatility."""
    realized = raw_strategy_returns.rolling(vol_window).std() * np.sqrt(TRADING_DAYS)
    leverage = (target_vol / realized.shift(1)).clip(lower=0.0, upper=max_leverage)
    leverage = leverage.replace([np.inf, -np.inf], np.nan).fillna(1.0)
    return base_weights.mul(leverage, axis=0), leverage


def backtest(
    prices: pd.DataFrame,
    lookback: int = 126,
    skip: int = 21,
    vol_window: int = 63,
    target_vol: float = 0.10,
    max_leverage: float = 2.0,
    cost_bps: float = 5.0,
    rebalance_frequency: str = "monthly",
):
    """Run a strictly lagged backtest with turnover-dependent transaction costs."""
    asset_returns = prices.pct_change().fillna(0.0)

    signal_weights = cross_sectional_weights(
        prices,
        lookback=lookback,
        skip=skip,
        vol_window=vol_window,
        rebalance_frequency=rebalance_frequency,
    )

    executed_weights = signal_weights.shift(1).fillna(0.0)
    raw_returns = (executed_weights * asset_returns).sum(axis=1)

    scaled_weights, leverage = apply_vol_target(
        raw_returns,
        executed_weights,
        target_vol=target_vol,
        vol_window=vol_window,
        max_leverage=max_leverage,
    )

    turnover = scaled_weights.diff().abs().sum(axis=1).fillna(0.0)
    gross_returns = (scaled_weights * asset_returns).sum(axis=1)
    costs = turnover * (cost_bps / 10000.0)
    net_returns = gross_returns - costs

    return {
        "weights": scaled_weights,
        "gross_returns": gross_returns,
        "net_returns": net_returns,
        "turnover": turnover,
        "costs": costs,
        "leverage": leverage,
    }


def max_drawdown(returns: pd.Series) -> float:
    equity = (1.0 + returns.fillna(0.0)).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    return float(drawdown.min())


def performance_metrics(returns: pd.Series, turnover: pd.Series | None = None) -> dict:
    r = returns.dropna()
    if len(r) == 0:
        raise ValueError("No returns available")

    total = (1.0 + r).prod()
    ann_ret = total ** (TRADING_DAYS / len(r)) - 1.0
    ann_vol = r.std() * np.sqrt(TRADING_DAYS)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan

    out = {
        "annualized_return": float(ann_ret),
        "annualized_volatility": float(ann_vol),
        "sharpe_ratio": float(sharpe),
        "max_drawdown": max_drawdown(r),
        "cumulative_return": float(total - 1.0),
    }
    if turnover is not None:
        aligned = turnover.reindex(r.index).fillna(0.0)
        out["average_daily_turnover"] = float(aligned.mean())
        out["annualized_turnover"] = float(aligned.mean() * TRADING_DAYS)
    return out


## Download market data

In [ ]:
UNIVERSE = ['XLB','XLC','XLE','XLF','XLI','XLK','XLP','XLRE','XLU','XLV','XLY']
raw = yf.download(UNIVERSE, start='2017-01-01', auto_adjust=True, progress=False, threads=True)
prices = raw['Close'].dropna(how='all').ffill().dropna()
spy_raw = yf.download('SPY', start='2017-01-01', auto_adjust=True, progress=False)
spy = spy_raw['Close']
if isinstance(spy, pd.DataFrame): spy = spy.iloc[:, 0]
spy = spy.reindex(prices.index).ffill().dropna()
prices = prices.reindex(spy.index).ffill().dropna()
prices.tail()


## Baseline backtest

In [ ]:
bt = backtest(prices, lookback=126, skip=21, vol_window=63, target_vol=0.10, max_leverage=2.0, cost_bps=5, rebalance_frequency='monthly')
spy_returns = spy.pct_change().fillna(0.0)
summary = pd.DataFrame({
    'strategy_gross': performance_metrics(bt['gross_returns'], bt['turnover']),
    'strategy_net': performance_metrics(bt['net_returns'], bt['turnover']),
    'SPY': performance_metrics(spy_returns),
})
summary.round(4)


In [ ]:
equity = pd.DataFrame({
    'strategy_net': (1 + bt['net_returns']).cumprod(),
    'SPY': (1 + spy_returns).cumprod(),
})
equity.plot(figsize=(11,5), title='Market-neutral momentum vs SPY')
plt.ylabel('Growth of $1')
plt.show()


## Chronological out-of-sample check

In [ ]:
periods = pd.DataFrame({
    'train_2017_2023': performance_metrics(bt['net_returns'].loc[:'2023-12-31']),
    'test_2024_present': performance_metrics(bt['net_returns'].loc['2024-01-01':]),
})
periods.round(4)


## Sensitivity analysis

In [ ]:
rows = []
for lookback in [63, 126, 189, 252]:
    for cost_bps in [0, 5, 10, 20]:
        tmp = backtest(prices, lookback=lookback, skip=21, vol_window=63, target_vol=0.10, max_leverage=2.0, cost_bps=cost_bps, rebalance_frequency='monthly')
        m = performance_metrics(tmp['net_returns'], tmp['turnover'])
        rows.append({'lookback': lookback, 'cost_bps': cost_bps, **m})
sensitivity = pd.DataFrame(rows)
sensitivity[['lookback','cost_bps','annualized_return','sharpe_ratio','max_drawdown','annualized_turnover']].round(4)


## Interpretation
This notebook is a research workflow rather than evidence of a live profitable trading system. The main checks are causality, implementation costs, robustness across parameter choices and chronological out-of-sample behaviour.